# Aurora-BP — Exploratory Data Analysis (EDA)

**Goal of this notebook:** understand the data *before* building any model, and produce one clean
index file (`measurement_index.csv`) that the modelling code will use later.

**Our plan (for context):** subject-level splits → deep-learning regressor on **ECG + PPG** →
predict SBP and DBP → post-hoc BP-category classification.

**Questions this EDA answers**
1. What files do we have, and are any waveform files missing?
2. Who are the participants (protocol, age, gender, skin tone, hypertension status)?
3. How many recordings does each subject have, in which phase/posture?
4. What do the BP labels look like? How balanced are the BP classes? How many readings sit near a class boundary?
5. How much does BP change *within* a person vs *between* people? (the key question for subject-split models)
6. What do raw ECG and PPG look like? Sampling rate, recording length, missing values, flat lines?
7. Where is the cuff reading inside each recording (what does `t` mean)?
8. How many usable model windows (e.g., 10 s / 30 s) do we get per subject after quality filters?
9. Does a subject-level train/val/test split keep the label distributions similar?

In [ ]:
# ===================== CONFIG — edit these =====================
DATA_DIR = "data/AuroraBP"           # Aurora-BP data (read only, never written to)
OUT_DIR = "eda_outputs"  # all EDA tables and figures go here

MAX_WAVEFORM_FILES = 300    # first pass: 300 files (~1-2 min). Set to None to scan ALL ~38k files.
N_WORKERS = 16              # parallel CPU workers for the waveform scan (shared server: don't use all 80)
QUALITY_THRESHOLD = 0.65    # the Aurora-BP paper used 0.65 as the inclusion threshold
WINDOW_SECONDS = [10, 30]   # candidate model input lengths to check
MAX_NAN_FRACTION = 0.05     # a window is "usable" if ECG and PPG each have < 5% missing samples
FS_EXPECTED = 500           # docs: all streams resampled to 500 Hz
RANDOM_SEED = 42
# ================================================================

In [ ]:
import os, glob, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

DATA = Path(DATA_DIR)
OUT = Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG / f"{name}.png", bbox_inches="tight")
    plt.show()

try:
    display
except NameError:          # running outside Jupyter
    display = print

def read_tsv(path, **kw):
    return pd.read_csv(path, sep="\t", **kw)

## 1. File inventory

In [ ]:
top_files = sorted(p.name for p in DATA.glob("*.tsv"))
print("Top-level TSV files:", top_files)

all_tsv = list(DATA.rglob("*.tsv"))
waveform_files = [p for p in all_tsv if p.parent != DATA]
print(f"Waveform files found on disk: {len(waveform_files):,}")
total_gb = sum(p.stat().st_size for p in waveform_files) / 1e9
print(f"Total waveform size: {total_gb:.2f} GB")

# index: file name -> full path (robust to how waveform_file_path is written)
wave_by_name = {p.name: p for p in waveform_files}

## 2. Load tables

In [ ]:
participants = read_tsv(DATA / "participants.tsv")
participants["protocol"] = participants["pid"].str[0].map({"a": "auscultatory", "o": "oscillometric"})

meas_list = []
for proto in ["auscultatory", "oscillometric"]:
    f = DATA / f"measurements_{proto}.tsv"
    if f.exists():
        m = read_tsv(f); m["protocol"] = proto; meas_list.append(m)
    else:
        print(f"WARNING: {f.name} not found")
meas = pd.concat(meas_list, ignore_index=True)

features = read_tsv(DATA / "features.tsv") if (DATA / "features.tsv").exists() else None

print("participants:", participants.shape)
print("measurements:", meas.shape)
print("features:", None if features is None else features.shape)
display(meas.head())

In [ ]:
def resolve_wave_path(row):
    """Find the waveform file for a measurement row, whatever form waveform_file_path takes."""
    raw = str(row.get("waveform_file_path", ""))
    for cand in [DATA / raw, DATA / raw.lstrip("/\\")]:
        if raw and cand.is_file():
            return cand
    name = Path(raw).name
    if name in wave_by_name:
        return wave_by_name[name]
    # fallback: rebuild the documented name <pid>.<phase>.<measurement>.tsv
    guess = f"{row['pid']}.{row['phase']}.{str(row['measurement']).replace(' ', '_')}.tsv"
    return wave_by_name.get(guess)

meas["wave_path"] = meas.apply(resolve_wave_path, axis=1)
meas["wave_exists"] = meas["wave_path"].notna()

print("Measurements with a waveform file:", meas["wave_exists"].sum(), "/", len(meas))
if "waveforms_generated" in meas:
    print(pd.crosstab(meas["waveforms_generated"], meas["wave_exists"],
                      rownames=["waveforms_generated"], colnames=["file found"]))

## 3. Participants

In [ ]:
print(participants["protocol"].value_counts(), "\n")
demo_cols = [c for c in ["age", "height", "weight", "fitzpatrick_scale"] if c in participants]
display(participants.groupby("protocol")[demo_cols].describe().T.round(1))

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, col in zip(axes, ["age", "gender", "fitzpatrick_scale", "self_report_htn"]):
    if col not in participants: ax.set_visible(False); continue
    if participants[col].dtype.kind in "fi" and participants[col].nunique() > 10:
        for proto, g in participants.groupby("protocol"):
            ax.hist(g[col].dropna(), bins=20, alpha=0.6, label=proto)
        ax.legend(fontsize=8)
    else:
        participants.groupby(["protocol", col]).size().unstack(0).plot.bar(ax=ax, rot=0)
    ax.set_title(col)
savefig("participants_overview")

print("Missing values per column (participants):")
print(participants.isna().sum()[lambda s: s > 0])

## 4. Recordings per subject

In [ ]:
# Posture / condition from measurement name (helps decide what to include, e.g. post-exercise)
def condition(name):
    n = str(name).lower()
    if "exercise" in n or "walk" in n or "run" in n or "cool" in n: return "post-exercise"
    if "calibration" in n: return "calibration"
    if "supine" in n: return "supine"
    if "sit" in n: return "sitting"
    if "stand" in n: return "standing"
    if "ambulatory" in n: return "ambulatory"
    return "other"

meas["condition"] = meas["measurement"].map(condition)
print(pd.crosstab(meas["protocol"], meas["phase"], margins=True), "\n")
print(pd.crosstab(meas["phase"], meas["condition"], margins=True))
print("\nMost common measurement names:")
print(meas["measurement"].value_counts().head(25))

In [ ]:
per_subj = (meas.groupby(["protocol", "pid"])
                .agg(n_meas=("measurement", "size"),
                     n_with_wave=("wave_exists", "sum"),
                     n_ambulatory=("phase", lambda s: (s == "ambulatory").sum()),
                     total_minutes=("duration", lambda s: s.sum() / 60))
                .reset_index())
display(per_subj.groupby("protocol")[["n_meas", "n_with_wave", "n_ambulatory", "total_minutes"]].describe().T.round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for proto, g in per_subj.groupby("protocol"):
    axes[0].hist(g["n_with_wave"], bins=30, alpha=0.6, label=proto)
    axes[1].hist(g["total_minutes"], bins=30, alpha=0.6, label=proto)
axes[0].set_title("Recordings with waveform per subject"); axes[0].set_xlabel("count")
axes[1].set_title("Total recorded minutes per subject"); axes[1].set_xlabel("minutes")
axes[0].legend(); savefig("recordings_per_subject")

for k in [3, 5, 10]:
    print(f"Subjects with fewer than {k} recordings: {(per_subj['n_with_wave'] < k).sum()}")

# Sanity check vs participants.tsv counts
chk = per_subj.merge(participants[["pid", "n_meas_inlab", "n_meas_ambulatory"]], on="pid", how="left")
chk["expected"] = chk["n_meas_inlab"].fillna(0) + chk["n_meas_ambulatory"].fillna(0)
print("Subjects where measurement count != participants.tsv count:", (chk["n_meas"] != chk["expected"]).sum())

## 5. Labels: SBP / DBP

In [ ]:
display(meas.groupby(["protocol", "phase"])[["sbp", "dbp"]].describe().round(1))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for proto, g in meas.groupby("protocol"):
    axes[0].hist(g["sbp"].dropna(), bins=40, alpha=0.6, label=proto)
    axes[1].hist(g["dbp"].dropna(), bins=40, alpha=0.6, label=proto)
    axes[2].scatter(g["sbp"], g["dbp"], s=6, alpha=0.4, label=proto)
axes[0].set_title("SBP (mmHg)"); axes[1].set_title("DBP (mmHg)"); axes[2].set_title("SBP vs DBP")
axes[2].set_xlabel("SBP"); axes[2].set_ylabel("DBP")
for a in axes: a.legend(fontsize=8)
savefig("bp_distributions")

# Physiologically suspicious labels
bad = meas[(meas.sbp < 70) | (meas.sbp > 220) | (meas.dbp < 35) | (meas.dbp > 140) | (meas.sbp <= meas.dbp)]
print(f"Suspicious BP rows: {len(bad)}"); display(bad[["pid", "phase", "measurement", "sbp", "dbp"]].head(20))

In [ ]:
# Auscultatory only: how much do the two human observers disagree? (= noise floor of the label)
if "consensus_systolic_error" in meas:
    a = meas[meas.protocol == "auscultatory"]
    print("Observer disagreement (absolute, mmHg):")
    display(a[["consensus_systolic_error", "consensus_diastolic_error"]].abs().describe().round(2))

## 6. BP classes (for the post-hoc classification)

In [ ]:
CLASS_ORDER = ["normal", "elevated", "stage1", "stage2"]

def bp_class(sbp, dbp):
    if pd.isna(sbp) or pd.isna(dbp): return np.nan
    if sbp >= 140 or dbp >= 90: return "stage2"
    if sbp >= 130 or dbp >= 80: return "stage1"
    if sbp >= 120: return "elevated"
    return "normal"

meas["bp_class"] = [bp_class(s, d) for s, d in zip(meas.sbp, meas.dbp)]
print("Class counts (per measurement):")
print(pd.crosstab(meas["bp_class"], meas["protocol"], margins=True).reindex(CLASS_ORDER + ["All"]))

# Distance to the nearest class threshold -> how many labels could flip from small measurement noise
def boundary_distance(sbp, dbp):
    return min([abs(sbp - t) for t in (120, 130, 140)] + [abs(dbp - t) for t in (80, 90)])
meas["dist_to_boundary"] = [boundary_distance(s, d) for s, d in zip(meas.sbp, meas.dbp)]
for k in [2, 5]:
    print(f"Readings within {k} mmHg of a class boundary: {(meas.dist_to_boundary <= k).mean():.1%}")

# How many classes does each subject visit? Subjects stuck in one class = the model may learn identity, not BP.
cls_per_subj = meas.groupby("pid")["bp_class"].nunique()
print("\nNumber of distinct BP classes per subject:")
print(cls_per_subj.value_counts().sort_index())

## 7. Within-subject vs between-subject BP variation
If most variation is *between* people, a model can look good just by recognising the person.
If within-person variation is small, predicting changes (delta BP) is the hard and honest task.

In [ ]:
def variance_split(df, col):
    subj_mean = df.groupby("pid")[col].transform("mean")
    within = ((df[col] - subj_mean) ** 2).mean()
    total = ((df[col] - df[col].mean()) ** 2).mean()
    return within, total

rows = []
for proto, g in meas.groupby("protocol"):
    for col in ["sbp", "dbp"]:
        w, t = variance_split(g.dropna(subset=[col]), col)
        rows.append(dict(protocol=proto, label=col, within_std=np.sqrt(w), total_std=np.sqrt(t),
                         within_share=w / t))
display(pd.DataFrame(rows).round(2))

subj_std = meas.groupby(["protocol", "pid"])["sbp"].std().reset_index()
fig, ax = plt.subplots(figsize=(7, 3.5))
for proto, g in subj_std.groupby("protocol"):
    ax.hist(g["sbp"].dropna(), bins=25, alpha=0.6, label=proto)
ax.set_title("Within-subject SBP standard deviation"); ax.set_xlabel("mmHg"); ax.legend()
savefig("within_subject_sbp_std")

## 8. Waveform scan (ECG + PPG)
Reads every waveform file once (or `MAX_WAVEFORM_FILES` of them) and records length, sampling rate,
missing values, flat lines, and the time range of `t`. This can take a while on the full dataset.

In [ ]:
SIGNALS = ["ekg", "optical"]

def longest_flat_run(x, tol=1e-9):
    d = np.abs(np.diff(x)) < tol
    if not d.any(): return 0
    runs = np.diff(np.flatnonzero(np.diff(np.r_[0, d.astype(int), 0])))[::2]
    return int(runs.max()) + 1 if len(runs) else 0

def count_usable_windows(df, win_s, fs):
    """Non-overlapping windows; usable = every signal has < MAX_NAN_FRACTION missing samples."""
    n = int(round(win_s * fs)); total = len(df) // n
    if total == 0:
        return 0, 0
    ok = np.ones(total, dtype=bool)
    for c in SIGNALS:
        if c not in df: return total, 0
        nan_frac = df[c].isna().to_numpy()[: total * n].reshape(total, n).mean(axis=1)
        ok &= nan_frac < MAX_NAN_FRACTION
    return total, int(ok.sum())

def scan_file(path):
    df = read_tsv(path, usecols=lambda c: c in ["t"] + SIGNALS)
    out = {"n_samples": len(df)}
    t = df["t"].to_numpy()
    dt = np.diff(t)
    out["t_min"], out["t_max"] = float(np.nanmin(t)), float(np.nanmax(t))
    out["fs_est"] = float(1 / np.nanmedian(dt)) if len(dt) else np.nan
    out["n_time_gaps"] = int((dt > 1.5 / FS_EXPECTED).sum()) if len(dt) else 0
    out["rec_seconds"] = out["t_max"] - out["t_min"]
    for c in SIGNALS:
        x = df[c].to_numpy(dtype=float) if c in df else np.full(len(df), np.nan)
        out[f"{c}_nan_frac"] = float(np.isnan(x).mean())
        xv = x[~np.isnan(x)]
        out[f"{c}_std"] = float(xv.std()) if len(xv) else np.nan
        out[f"{c}_flat_sec"] = longest_flat_run(xv) / FS_EXPECTED if len(xv) else np.nan
    fs = out["fs_est"] if np.isfinite(out["fs_est"]) else FS_EXPECTED
    for w in WINDOW_SECONDS:
        out[f"win{w}_total"], out[f"win{w}_usable"] = count_usable_windows(df, w, fs)
    return out

to_scan = meas[meas.wave_exists]
if MAX_WAVEFORM_FILES:
    to_scan = to_scan.sample(min(MAX_WAVEFORM_FILES, len(to_scan)), random_state=RANDOM_SEED)

from concurrent.futures import ProcessPoolExecutor
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **k: x

def scan_one(item):
    idx, path = item
    try:
        r = scan_file(path); r["idx"] = idx
    except Exception as e:
        r = {"idx": idx, "scan_error": str(e)}
    return r

items = list(zip(to_scan.index, to_scan["wave_path"]))
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    scan_rows = list(tqdm(ex.map(scan_one, items, chunksize=8), total=len(items), desc="scanning waveforms"))

scan = pd.DataFrame(scan_rows).set_index("idx")
meas = meas.join(scan, how="left")
print("Files scanned:", len(scan), "| errors:", scan.get("scan_error", pd.Series(dtype=str)).notna().sum())

In [ ]:
cols = ["rec_seconds", "fs_est", "n_time_gaps", "ekg_nan_frac", "optical_nan_frac",
        "ekg_flat_sec", "optical_flat_sec", "t_min", "t_max"]
display(meas.groupby(["protocol", "phase"])[cols].median().round(3))

print("Estimated sampling rates (rounded):"); print(meas["fs_est"].round().value_counts().head())
print("\nRecording length vs `duration` column (seconds):")
print((meas["rec_seconds"] - meas["duration"]).describe().round(2))

fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for proto, g in meas.groupby("protocol"):
    axes[0].hist(g["rec_seconds"].dropna(), bins=40, alpha=0.6, label=proto)
    axes[1].hist(g["ekg_nan_frac"].dropna(), bins=40, alpha=0.6, label=proto)
    axes[2].hist(g["optical_nan_frac"].dropna(), bins=40, alpha=0.6, label=proto)
axes[0].set_title("Recording length (s)"); axes[1].set_title("ECG missing fraction")
axes[2].set_title("PPG missing fraction")
for a in axes: a.legend(fontsize=8)
savefig("waveform_quality")

### Where is the cuff reading inside the recording?
Docs: `t` is time relative to `date_time`. If `t` starts negative, `date_time` is probably the cuff
moment and the reading sits at t = 0. If `t` always starts at 0, `date_time` is the recording start.
This decides which window we cut for the model.

In [ ]:
print(meas.groupby(["protocol", "phase"])[["t_min", "t_max"]].describe().round(1))
print("\nShare of recordings with negative t_min:", (meas["t_min"] < 0).mean().round(3))

## 9. Look at raw ECG + PPG

In [ ]:
def plot_example(row, seconds=10, start=None, tag=""):
    df = read_tsv(row["wave_path"], usecols=lambda c: c in ["t"] + SIGNALS)
    t0 = df["t"].iloc[0] if start is None else start
    seg = df[(df.t >= t0) & (df.t < t0 + seconds)]
    fig, axes = plt.subplots(2, 1, figsize=(14, 4), sharex=True)
    axes[0].plot(seg.t, seg.ekg, lw=0.7); axes[0].set_ylabel("ECG (mV)")
    axes[1].plot(seg.t, seg.optical, lw=0.7, color="tab:green"); axes[1].set_ylabel("PPG (a.u.)")
    axes[1].set_xlabel("t (s)")
    fig.suptitle(f"{row['pid']} | {row['phase']} | {row['measurement']} | SBP {row['sbp']} / DBP {row['dbp']}")
    savefig(f"example_{row['pid']}_{row['phase']}_{str(row['measurement']).replace(' ', '_')}{tag}")

examples = meas[meas.wave_exists].groupby(["protocol", "phase"]).head(1)
for _, r in examples.iterrows():
    plot_example(r, seconds=10)

# Zoom: 3 seconds, to see the delay between the ECG spike and the PPG pulse (pulse arrival time)
plot_example(examples.iloc[0], seconds=3, tag="_zoom3s")

## 10. Model-window yield per subject

In [ ]:
for w in WINDOW_SECONDS:
    meas[f"win{w}_usable_q"] = np.where(meas["optical_quality"] >= QUALITY_THRESHOLD,
                                        meas[f"win{w}_usable"], 0)

agg = {f"win{w}_usable": "sum" for w in WINDOW_SECONDS}
agg.update({f"win{w}_usable_q": "sum" for w in WINDOW_SECONDS})
win_subj = meas.groupby(["protocol", "pid"]).agg(agg).reset_index()
display(win_subj.groupby("protocol").describe().T.round(1))

for w in WINDOW_SECONDS:
    print(f"{w}s windows | all: {int(meas[f'win{w}_usable'].sum()):,} "
          f"| with optical_quality >= {QUALITY_THRESHOLD}: {int(meas[f'win{w}_usable_q'].sum()):,} "
          f"| subjects with 0 usable (quality-filtered): {(win_subj[f'win{w}_usable_q'] == 0).sum()}")

print("\nShare of measurements passing optical quality:",
      (meas["optical_quality"] >= QUALITY_THRESHOLD).mean().round(3))

## 11. Quick sanity check with features.tsv
Pulse arrival time (rPAT) should go *down* when BP goes *up*. If this correlation is visible,
the ECG + PPG signal really carries BP information.

In [ ]:
if features is not None and "rpat_optical" in features:
    f = features[features["phase"] != "synthetic"].dropna(subset=["rpat_optical", "sbp"])
    print("Pearson r (rpat_optical vs sbp), all rows:", round(f["rpat_optical"].corr(f["sbp"]), 3))
    within = f.groupby("pid").apply(lambda g: g["rpat_optical"].corr(g["sbp"]) if len(g) > 4 else np.nan)
    print("Median within-subject r:", round(within.median(), 3))
    plt.figure(figsize=(5, 4)); plt.scatter(f["rpat_optical"], f["sbp"], s=5, alpha=0.4)
    plt.xlabel("rPAT optical (s)"); plt.ylabel("SBP (mmHg)"); savefig("rpat_vs_sbp")

## 12. Subject-level split preview

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

usable = meas[meas.wave_exists & meas.sbp.notna()].copy()
usable["split"] = "train"

# Split each protocol separately so train/val/test all contain both protocols (70 / 15 / 15 by subject)
for proto, g in usable.groupby("protocol"):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_SEED)
    _, tmp_i = next(gss.split(g, groups=g.pid))
    tmp = g.iloc[tmp_i]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_SEED)
    va_i, te_i = next(gss2.split(tmp, groups=tmp.pid))
    usable.loc[tmp.index[va_i], "split"] = "val"
    usable.loc[tmp.index[te_i], "split"] = "test"

assert usable.groupby("pid")["split"].nunique().max() == 1, "a subject appears in more than one split!"
display(usable.groupby("split").agg(subjects=("pid", "nunique"), recordings=("pid", "size"),
                                    sbp_mean=("sbp", "mean"), sbp_std=("sbp", "std"),
                                    dbp_mean=("dbp", "mean")).round(1))
print(pd.crosstab(usable["split"], usable["bp_class"], normalize="index").reindex(columns=CLASS_ORDER).round(3))
print(pd.crosstab(usable["split"], usable["protocol"]))

## 13. Save the measurement index (input for the modelling code)

In [ ]:
meas = meas.merge(usable[["split"]], left_index=True, right_index=True, how="left")
keep = ["pid", "protocol", "phase", "measurement", "condition", "date_time", "sbp", "dbp", "bp_class",
        "dist_to_boundary", "duration", "optical_quality", "pressure_quality", "wave_exists", "wave_path",
        "rec_seconds", "fs_est", "t_min", "t_max", "ekg_nan_frac", "optical_nan_frac",
        "ekg_flat_sec", "optical_flat_sec"] + \
       [f"win{w}_usable" for w in WINDOW_SECONDS] + [f"win{w}_usable_q" for w in WINDOW_SECONDS] + ["split"]
keep = [c for c in keep if c in meas]
index = meas[keep].merge(participants.drop(columns=["protocol"]), on="pid", how="left")
index.to_csv(OUT / "measurement_index.csv", index=False)
per_subj.merge(win_subj, on=["protocol", "pid"], how="left").to_csv(OUT / "subject_summary.csv", index=False)
print("Saved:", OUT / "measurement_index.csv", "and", OUT / "subject_summary.csv")
print("Figures in:", FIG)